In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# 1. Define the Classic Golf Dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 
                'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temp': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 
             'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 
                 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 
             'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 
             'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
}

df = pd.DataFrame(data)

# 2. Preprocessing: Encode categorical strings to integers for CART
# Note: In a real pipeline you might use OneHotEncoding, but LabelEncoding 
# keeps the tree smaller and closer to the classic textbook example.
le_dict = {}
for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

X = df.drop('Play', axis=1)
y = df['Play']
feature_names = X.columns.tolist()

# 3. Initialize and Train the Classifier
clf = DecisionTreeClassifier(criterion='entropy', random_state=42)
clf.fit(X, y)

# 4. Custom Function: Unpacking the Engineering Logic (from previous execution)
def print_tree_logic(tree_model, feature_names, label_encoders):
    tree = tree_model.tree_
    
    def recurse(node, depth):
        indent = "  " * depth
        if tree.children_left[node] != tree.children_right[node]:
            left, right = tree.children_left[node], tree.children_right[node]
            
            feature_idx = tree.feature[node]
            feature = feature_names[feature_idx]
            threshold = tree.threshold[node]
            parent_entropy = tree.impurity[node]
            n_parent = tree.n_node_samples[node]
            
            left_entropy = tree.impurity[left]
            right_entropy = tree.impurity[right]
            
            # Information Gain Calculation
            weighted_child_entropy = ((tree.n_node_samples[left] / n_parent) * left_entropy) + \
                                     ((tree.n_node_samples[right] / n_parent) * right_entropy)
            info_gain = parent_entropy - weighted_child_entropy
            
            # Decode the threshold back to human context where possible
            # (CART splits floats, e.g., <= 0.5. If 0 is Overcast and 1 is Rain, it splits them)
            print(f"{indent}Decision Node: Split on '{feature}' <= {threshold:.2f}")
            print(f"{indent}  ├─ Node Entropy: {parent_entropy:.4f} ({n_parent} samples)")
            print(f"{indent}  ├─ Information Gain: {info_gain:.4f}")
            
            recurse(left, depth + 1)
            recurse(right, depth + 1)
        else:
            entropy = tree.impurity[node]
            samples = tree.n_node_samples[node]
            # Determine predicted class
            predicted_class_idx = np.argmax(tree.value[node])
            predicted_class = label_encoders['Play'].inverse_transform([predicted_class_idx])[0]
            
            print(f"{indent}Leaf Node (Predict: {predicted_class}):")
            print(f"{indent}  └─ Final Entropy: {entropy:.4f} ({samples} samples)\n")

    print("--- Binary (CART) Decision Tree Logic & Information Gain ---\n")
    recurse(0, 0)
    print("----------------------------------------------------------")

# Execute the trace
print_tree_logic(clf, feature_names, le_dict)

# 5. Generate the Visual Tree Diagram
plt.figure(figsize=(12, 8))
plot_tree(
    clf, 
    filled=True, 
    feature_names=feature_names, 
    class_names=le_dict['Play'].classes_, 
    rounded=True,
    fontsize=12
)
plt.title("Golf Dataset: Binary Split Architecture (CART)", fontsize=16)
plt.show()

--- Binary (CART) Decision Tree Logic & Information Gain ---

Decision Node: Split on 'Outlook' <= 0.50
  ├─ Node Entropy: 0.9403 (14 samples)
  ├─ Information Gain: 0.2260
  Leaf Node (Predict: Yes):
    └─ Final Entropy: 0.0000 (4 samples)

  Decision Node: Split on 'Humidity' <= 0.50
    ├─ Node Entropy: 1.0000 (10 samples)
    ├─ Information Gain: 0.2781
    Decision Node: Split on 'Outlook' <= 1.50
      ├─ Node Entropy: 0.7219 (5 samples)
      ├─ Information Gain: 0.3219
      Decision Node: Split on 'Wind' <= 0.50
        ├─ Node Entropy: 1.0000 (2 samples)
        ├─ Information Gain: 1.0000
        Leaf Node (Predict: No):
          └─ Final Entropy: 0.0000 (1 samples)

        Leaf Node (Predict: Yes):
          └─ Final Entropy: 0.0000 (1 samples)

      Leaf Node (Predict: No):
        └─ Final Entropy: 0.0000 (3 samples)

    Decision Node: Split on 'Wind' <= 0.50
      ├─ Node Entropy: 0.7219 (5 samples)
      ├─ Information Gain: 0.3219
      Decision Node: Split on 'Te